In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 0: Install & Import Dependencies

In [2]:
import re
import pandas as pd
import numpy as np

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


## Helper Function for Text Cleaning

All helper functions are implemented below, carried over from the Text Preprocessing Notebook.

In [3]:
# ── Helper 1: Lowercase ──────────────────────────────────────────────────────
def lower_order(text):
  """
  Converts all characters in the input text to lowercase.
  Input Args:
    text: input string.
  Returns:
    Lowercase version of the input string.
  """
  return text.lower()


# ── Helper 2: Remove URLs ─────────────────────────────────────────────────────
def remove_urls(text):
  """
  Removes URLs from a string using regex.
  Input Args:
    text: string that may contain URLs.
  Returns:
    String with URLs removed.
  """
  url_pattern = re.compile(r'https?://\S+|www\.\S+')
  return url_pattern.sub(r'', text)


# ── Helper 3: Remove Emojis ───────────────────────────────────────────────────
def remove_emoji(string):
  """
  Replaces emojis in a string with a whitespace.
  Input Args:
    string: input string that may contain emojis.
  Returns:
    String with emojis replaced by spaces.
  """
  emoji_pattern = re.compile("["
                             u"\U0001F600-\U0001F64F"   # emoticons
                             u"\U0001F300-\U0001F5FF"   # symbols & pictographs
                             u"\U0001F680-\U0001F6FF"   # transport & map symbols
                             u"\U0001F1E0-\U0001F1FF"   # flags (iOS)
                             u"\U00002702-\U000027B0"
                             u"\U000024C2-\U0001F251"
                             "]+", flags=re.UNICODE)
  return emoji_pattern.sub(r' ', string)


# ── Helper 4: Remove Unwanted Characters ─────────────────────────────────────
def removeunwanted_characters(document):
  """
  Removes mentions, hashtags, punctuation, emojis, and extra spaces.
  Input Args:
    document: a string of text.
  Returns:
    Cleaned string.
  """
  # Remove user mentions (@username)
  document = re.sub("@[A-Za-z0-9_]+", " ", document)
  # Remove hashtags (#tag)
  document = re.sub("#[A-Za-z0-9_]+", "", document)
  # Remove punctuation and special characters (keep alphanumeric + spaces)
  document = re.sub("[^0-9A-Za-z ]", "", document)
  # Remove any remaining emojis
  document = remove_emoji(document)
  # Collapse multiple spaces into a single space
  document = re.sub(r'\s+', ' ', document)
  return document.strip()


# ── Helper 5: Remove Stopwords ────────────────────────────────────────────────
def remove_stopwords(tokens):
  """
  Removes English stopwords from a list of tokens.
  Input Args:
    tokens: list of word tokens.
  Returns:
    List of tokens with stopwords removed.
  """
  stop_words = set(stopwords.words('english'))
  return [word for word in tokens if word not in stop_words]


# ── Helper 6: Lemmatization ───────────────────────────────────────────────────
def lemmatization(tokens):
  """
  Applies lemmatization to a list of tokens.
  Input Args:
    tokens: list of word tokens.
  Returns:
    List of lemmatized tokens.
  """
  lemmatizer = WordNetLemmatizer()
  return [lemmatizer.lemmatize(word) for word in tokens]


# ── Helper 7: Stemming ────────────────────────────────────────────────────────
def stemming(tokens):
  """
  Applies Porter Stemming to a list of tokens.
  Input Args:
    tokens: list of word tokens.
  Returns:
    List of stemmed tokens.
  """
  porter = PorterStemmer()
  return [porter.stem(word) for word in tokens]

# Build a Text Cleaning Pipeline

In [4]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
  """
  Compiles various text pre-processing steps into a single pipeline.
  Steps: lowercase → remove URLs → remove emojis → remove unwanted characters
         → tokenize → remove stopwords → lemmatize OR stem.

  Input Args:
    dataset (str): A string of text to be cleaned.
    rule (str)   : 'lemmatize' (default) or 'stem'.

  Returns:
    str: A single string of space-separated cleaned tokens.
  """
  # Convert the input to lowercase
  data = lower_order(dataset)

  # Remove URLs
  data = remove_urls(data)

  # Remove emojis
  data = remove_emoji(data)

  # Remove all other unwanted characters (mentions, hashtags, punctuation, extra spaces)
  data = removeunwanted_characters(data)

  # Tokenize
  tokens = word_tokenize(data)

  # Remove stopwords
  tokens = remove_stopwords(tokens)

  # Apply lemmatization or stemming
  if rule == "lemmatize":
    tokens = lemmatization(tokens)
  elif rule == "stem":
    tokens = stemming(tokens)
  else:
    print("Pick between 'lemmatize' or 'stem'")

  return " ".join(tokens)

### Quick Pipeline Test

In [5]:
# Quick sanity check
sample = "Hello @gabe_flomo 👋🏾, I still want us to hit that new sushi spot??? LMK when you're free! #sushiBros http://example.com"
print("Original :", sample)
print("Lemmatize:", text_cleaning_pipeline(sample, rule='lemmatize'))
print("Stem     :", text_cleaning_pipeline(sample, rule='stem'))

Original : Hello @gabe_flomo 👋🏾, I still want us to hit that new sushi spot??? LMK when you're free! #sushiBros http://example.com
Lemmatize: hello still want u hit new sushi spot lmk youre free
Stem     : hello still want us hit new sushi spot lmk your free


# Text Classification using Machine Learning Models

## Step 1 — Load the Dataset

In [6]:
# Update the path below to match your Google Drive location
df = pd.read_csv('/content/drive/MyDrive/AI ML/Data/trum_tweet_sentiment_analysis.csv')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (1850123, 2)
Columns: ['text', 'Sentiment']


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [7]:
# Check label distribution
print("Label distribution:")
print(df['Sentiment'].value_counts())

# Check for null values
print("\nNull values:")
print(df[['text', 'Sentiment']].isnull().sum())

Label distribution:
Sentiment
0    1244211
1     605912
Name: count, dtype: int64

Null values:
text         0
Sentiment    0
dtype: int64


In [8]:
# Drop rows with missing text or label
df = df[['text', 'Sentiment']].dropna().reset_index(drop=True)
print("Dataset size after dropping nulls:", df.shape)

Dataset size after dropping nulls: (1850123, 2)


## Step 2 — Text Cleaning and Tokenization

In [9]:
# Apply the full text cleaning pipeline to the 'text' column.
# This handles: lowercasing, URL removal, emoji removal, punctuation removal,
# stopword removal, and lemmatization.
print("Cleaning text ... (this may take a moment)")
df['cleaned_text'] = df['text'].apply(lambda x: text_cleaning_pipeline(x, rule='lemmatize'))

# Preview the result
print("\nSample before cleaning:")
print(df['text'][0])
print("\nSample after cleaning:")
print(df['cleaned_text'][0])

Cleaning text ... (this may take a moment)

Sample before cleaning:
RT @JohnLeguizamo: #trump not draining swamp but our taxpayer dollars on his trips to advertise his properties! @realDonaldTrump https://t.co/gFBvUkMX9z

Sample after cleaning:
rt draining swamp taxpayer dollar trip advertise property


## Step 3 — Train-Test Split

In [11]:
X = df['cleaned_text']   # Features: cleaned tweet text
y = df['Sentiment']          # Target: sentiment label

# 80% training, 20% testing  |  random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # keeps class proportions equal in both splits
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

Training samples : 1480098
Testing samples  : 370025


## Step 4 — TF-IDF Vectorization

TF-IDF (Term Frequency–Inverse Document Frequency) converts raw text into
numerical feature vectors that machine learning models can process.  
- **TF** rewards words that appear frequently in a single tweet.  
- **IDF** penalises words that appear in almost every tweet (less informative).

In [12]:
# Initialise the vectorizer
# max_features=10000  → keep only the top 10,000 most frequent terms
# ngram_range=(1,2)   → include both single words (unigrams) and
#                        two-word phrases (bigrams) for richer features
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

# Fit on training data ONLY to avoid data leakage, then transform both splits
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("TF-IDF training matrix shape:", X_train_tfidf.shape)
print("TF-IDF testing  matrix shape:", X_test_tfidf.shape)

TF-IDF training matrix shape: (1480098, 10000)
TF-IDF testing  matrix shape: (370025, 10000)


## Step 5 — Model Training and Evaluation

In [13]:
# ── Train: Logistic Regression ────────────────────────────────────────────────
# max_iter=1000 ensures the solver converges on larger/sparser TF-IDF matrices
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

print("Model training complete.")

Model training complete.


In [14]:
# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test_tfidf)

print("Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred))

Classification Report
              precision    recall  f1-score   support

           0       0.93      0.96      0.94    248842
           1       0.90      0.86      0.88    121183

    accuracy                           0.92    370025
   macro avg       0.92      0.91      0.91    370025
weighted avg       0.92      0.92      0.92    370025



## Step 6 (Bonus) — Predict on New Tweets

In [15]:
def predict_sentiment(tweet):
  """
  Takes a raw tweet string and returns its predicted sentiment label.
  """
  cleaned = text_cleaning_pipeline(tweet, rule='lemmatize')
  vectorized = tfidf.transform([cleaned])
  prediction = model.predict(vectorized)
  return prediction[0]

# Example predictions
test_tweets = [
    "America is doing great! The economy has never been stronger.",
    "Fake news media is totally corrupt and dishonest.",
    "Thank you to all our brave soldiers serving overseas!"
]

print(f"{'Tweet':<65} | {'Predicted Label'}")
print("-" * 85)
for tweet in test_tweets:
  label = predict_sentiment(tweet)
  print(f"{tweet[:64]:<65} | {label}")

Tweet                                                             | Predicted Label
-------------------------------------------------------------------------------------
America is doing great! The economy has never been stronger.      | 1
Fake news media is totally corrupt and dishonest.                 | 0
Thank you to all our brave soldiers serving overseas!             | 1
